# Point-in-Time Multi-Frequency Wasserstein Signal — Computation

**Reproducible construction of the signal candidates used in the article.**

This notebook computes the multi-frequency dispersion signal from the two validated NYSE point-in-time panels: large capitalisations above the NYSE P90 breakpoint and the NYSE P20–P50 segment. At each formation month, the exact same ex-ante membership is used as the column axis of the daily, weekly, and monthly return matrices. Missing returns are never imputed, columns are never cleaned independently by frequency, and no observation after the formation close is used.

The notebook writes resumable calculation caches only. It does **not** publish article inputs. Publication is deliberately delegated to `tests/02_validate_and_publish_pit_signals.ipynb`, which must pass every assertion first.

## 1. Environment and immutable configuration

The reference specification uses a 36-month estimation window, 50 free-support barycenter atoms, uniform frequency weights, square-root-of-horizon scaling, and sliced $W_2^2$. Robustness configurations alter exactly one construction choice at a time. The 48- and 60-month configurations use the same 60-month-eligible monthly universe, so their differences isolate estimation-window length rather than asset availability.

In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import platform
import time

import numpy as np
import pandas as pd
import polars as pl
import ot
from sklearn.cluster import KMeans

ROOT = Path.cwd()
if ROOT.name in {'notebooks', 'tests'}:
    ROOT = ROOT.parent

SOURCES = {
    'big_caps': ROOT / 'data' / 'processed' / 'nyse_big_caps_pit_daily.parquet',
    'small_caps_p20_p50': ROOT / 'data' / 'processed' / 'nyse_small_caps_p20_p50_pit_daily.parquet',
}
CACHE_DIR = ROOT / 'cache' / 'signal' / 'pit_engine'
CANDIDATE_PATH = ROOT / 'cache' / 'signal' / 'rho_pit_candidate.parquet'
AUDIT_PATH = ROOT / 'cache' / 'signal' / 'rho_pit_matrix_audit.parquet'

ENGINE_VERSION = 'pit-signal-v1.1.0'
BASE_SEED = 20250301
N_PROJECTIONS = 200
N_QUANTILES = 200
BARYCENTER_MAX_ITER = 30
BARYCENTER_TOL = 1e-4
CHECKPOINT_EVERY = 12
FORCE_RECOMPUTE = False
WRITE_CACHES = True

for path in SOURCES.values():
    assert path.exists(), path
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Engine: {ENGINE_VERSION}')
print(f'Python: {platform.python_version()} | Polars: {pl.__version__} | POT: {ot.__version__}')
print(f'Projections: {N_PROJECTIONS} | quantiles: {N_QUANTILES} | base seed: {BASE_SEED}')

In [ ]:
CONFIGS = [
    dict(config_id='reference_w36', lookback_months=36, M_atoms=50, frequency_weights='uniform', scaling='power', exponent=0.5, distance='sliced', barycenter='free_support'),
    dict(config_id='reference_w48', lookback_months=48, M_atoms=50, frequency_weights='uniform', scaling='power', exponent=0.5, distance='sliced', barycenter='free_support'),
    dict(config_id='reference_w60', lookback_months=60, M_atoms=50, frequency_weights='uniform', scaling='power', exponent=0.5, distance='sliced', barycenter='free_support'),
    dict(config_id='cardinality_m36', lookback_months=36, M_atoms=36, frequency_weights='uniform', scaling='power', exponent=0.5, distance='sliced', barycenter='free_support'),
    dict(config_id='cardinality_m72', lookback_months=36, M_atoms=72, frequency_weights='uniform', scaling='power', exponent=0.5, distance='sliced', barycenter='free_support'),
    dict(config_id='weights_tk', lookback_months=36, M_atoms=50, frequency_weights='sample_size', scaling='power', exponent=0.5, distance='sliced', barycenter='free_support'),
    dict(config_id='weights_log_tk', lookback_months=36, M_atoms=50, frequency_weights='log_sample_size', scaling='power', exponent=0.5, distance='sliced', barycenter='free_support'),
    dict(config_id='scaling_vol', lookback_months=36, M_atoms=50, frequency_weights='uniform', scaling='realized_volatility', exponent=np.nan, distance='sliced', barycenter='free_support'),
    dict(config_id='scaling_h04', lookback_months=36, M_atoms=50, frequency_weights='uniform', scaling='power', exponent=0.4, distance='sliced', barycenter='free_support'),
    dict(config_id='scaling_h06', lookback_months=36, M_atoms=50, frequency_weights='uniform', scaling='power', exponent=0.6, distance='sliced', barycenter='free_support'),
    dict(config_id='distance_exact', lookback_months=36, M_atoms=50, frequency_weights='uniform', scaling='power', exponent=0.5, distance='exact', barycenter='free_support'),
    dict(config_id='barycenter_1d', lookback_months=36, M_atoms=50, frequency_weights='uniform', scaling='power', exponent=0.5, distance='sliced', barycenter='quantile_1d'),
]

CONFIG_TABLE = pd.DataFrame(CONFIGS)
assert CONFIG_TABLE['config_id'].is_unique
display(CONFIG_TABLE)

## 2. Source contract and cache identity

A cache is reusable only when its engine version, source fingerprint, and full numerical configuration match. The fingerprint is intentionally conservative: replacing a source Parquet invalidates its associated caches. Atomic writes prevent an interrupted kernel from leaving a valid-looking partial file.

In [ ]:
REQUIRED_SOURCE_COLUMNS = {
    'PERMNO', 'DlyCalDt', 'DlyRet', 'is_formation_member',
    'formation_member_month', 'formation_member_active_month',
    'formation_member_rank', 'formation_member_strict_hist60',
}

def source_signature(path: Path) -> str:
    stat = path.stat()
    payload = f'{path.resolve()}|{stat.st_size}|{stat.st_mtime_ns}'
    return hashlib.sha256(payload.encode()).hexdigest()

def stable_digest(payload: dict) -> str:
    serialised = json.dumps(payload, sort_keys=True, separators=(',', ':'), allow_nan=True)
    return hashlib.sha256(serialised.encode()).hexdigest()

def atomic_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    df.to_parquet(temporary, index=False)
    temporary.replace(path)

def seed_for(universe: str, formation_month: pd.Timestamp) -> int:
    # Common random numbers across configurations at a given universe-date.
    key = f'{BASE_SEED}|{universe}|{formation_month:%Y-%m}'
    return int(hashlib.sha256(key.encode()).hexdigest()[:8], 16)

SOURCE_SIGNATURES = {}
for universe, path in SOURCES.items():
    schema = pl.scan_parquet(path).collect_schema()
    missing = REQUIRED_SOURCE_COLUMNS - set(schema.names())
    assert not missing, f'{universe}: missing columns {sorted(missing)}'
    SOURCE_SIGNATURES[universe] = source_signature(path)

print('Source contracts and fingerprints validated.')

## 3. PIT membership and aligned return panels

For formation month $M$, membership is read exclusively from the 100 formation-close rows marked `is_formation_member=1`. It is never inferred from subsequent observability in $M+1$. Returns are truncated at the recorded formation close. The window contains exactly $W$ calendar months, from $M-W+1$ through $M$. Weekly and monthly returns are compounded from the same full daily matrix; they are not separately filtered.

In [ ]:
def load_universe(path: Path):
    selected = (
        pl.scan_parquet(path)
        .filter(pl.col('is_formation_member') == 1)
        .select([
            'DlyCalDt', 'PERMNO',
            pl.col('formation_member_month').alias('formation_month'),
            pl.col('formation_member_active_month').alias('active_month'),
            pl.col('formation_member_rank').alias('rank'),
            pl.col('formation_member_strict_hist60').alias('strict_hist60'),
        ])
        .collect(engine='streaming').to_pandas()
    )
    selected['DlyCalDt'] = pd.to_datetime(selected['DlyCalDt'])
    selected['formation_month'] = pd.to_datetime(selected['formation_month'])
    selected['active_month'] = pd.to_datetime(selected['active_month'])
    assert selected['formation_month'].notna().all()
    assert selected['strict_hist60'].eq(True).all()
    assert selected['active_month'].eq(selected['formation_month'] + pd.offsets.MonthBegin(1)).all()
    assert not selected.duplicated(['formation_month', 'PERMNO']).any()
    assert not selected.duplicated(['formation_month', 'rank']).any()

    counts = selected.groupby('formation_month').agg(
        n=('PERMNO', 'size'), n_ranks=('rank', 'nunique'),
        min_rank=('rank', 'min'), max_rank=('rank', 'max'),
        first_date=('DlyCalDt', 'min'), last_date=('DlyCalDt', 'max'),
    )
    assert counts['n'].eq(100).all() and counts['n_ranks'].eq(100).all()
    assert counts['min_rank'].eq(1).all() and counts['max_rank'].eq(100).all()
    assert counts['first_date'].eq(counts['last_date']).all()

    memberships = {
        pd.Timestamp(month): group.sort_values(['rank', 'PERMNO'])['PERMNO'].astype(int).tolist()
        for month, group in selected.groupby('formation_month', sort=True)
    }
    month_end = counts['last_date'].to_dict()

    returns = pl.read_parquet(path, columns=['PERMNO', 'DlyCalDt', 'DlyRet'])
    duplicate_count = returns.select(pl.struct(['DlyCalDt', 'PERMNO']).is_duplicated().sum()).item()
    assert duplicate_count == 0, 'Duplicate PERMNO-date observations in source.'
    wide = returns.pivot(on='PERMNO', index='DlyCalDt', values='DlyRet').sort('DlyCalDt')
    pivot = wide.to_pandas().set_index('DlyCalDt').sort_index()
    pivot.index = pd.to_datetime(pivot.index)
    pivot.columns = pivot.columns.astype(int)
    return pivot, memberships, month_end

def compounded_returns(daily: pd.DataFrame, rule: str) -> pd.DataFrame:
    return (1.0 + daily).resample(rule).prod(min_count=1) - 1.0

def raw_aligned_matrices(pivot, assets, formation_month, month_end, lookback_months):
    formation_month = pd.Timestamp(formation_month)
    formation_date = pd.Timestamp(month_end[formation_month])
    start_month = formation_month - pd.DateOffset(months=lookback_months - 1)
    start_date = pd.Timestamp(start_month)

    assert len(assets) == 100 == len(set(assets))
    missing_columns = sorted(set(assets) - set(pivot.columns))
    assert not missing_columns, f'Missing PERMNO columns: {missing_columns[:5]}'

    daily = pivot.loc[(pivot.index >= start_date) & (pivot.index <= formation_date), assets].copy()
    expected_months = pd.period_range(start_month, formation_month, freq='M')
    observed_months = daily.index.to_period('M').unique().sort_values()
    assert observed_months.equals(expected_months), 'The calendar-month window is not contiguous.'
    assert daily.notna().all().all(), 'The selected PIT daily matrix is not full.'
    assert daily.index.max() == formation_date

    weekly = compounded_returns(daily, 'W-FRI')
    monthly = compounded_returns(daily, 'ME')
    assert list(daily.columns) == list(weekly.columns) == list(monthly.columns)
    assert weekly.notna().all().all() and monthly.notna().all().all()
    return daily, weekly, monthly, formation_date, start_date

## 4. Estimator

The free-support barycenter always assigns equal mass to the three empirical frequency distributions. Frequency-weight robustness changes only the final dispersion aggregation. Sliced configurations use common random directions across specifications at a given universe-date, which reduces Monte Carlo noise in pairwise comparisons without changing any marginal estimator.

In [ ]:
HORIZONS = np.array([1.0, 5.0, 21.0])

def scale_matrices(daily, weekly, monthly, config):
    arrays = [daily.to_numpy(float), weekly.to_numpy(float), monthly.to_numpy(float)]
    if config['scaling'] == 'power':
        exponent = float(config['exponent'])
        arrays = [x / (h ** exponent) for x, h in zip(arrays, HORIZONS)]
    elif config['scaling'] == 'realized_volatility':
        scaled = []
        for x in arrays:
            sigma = x.std(axis=0, ddof=1)
            assert np.isfinite(sigma).all() and (sigma > 1e-12).all(), 'Zero or invalid realized volatility.'
            scaled.append(x / sigma)
        arrays = scaled
    else:
        raise ValueError(config['scaling'])
    assert all(np.isfinite(x).all() for x in arrays)
    return arrays

def frequency_weights(arrays, mode):
    if mode == 'uniform':
        weights = np.ones(3)
    elif mode == 'sample_size':
        weights = np.array([len(x) for x in arrays], dtype=float)
    elif mode == 'log_sample_size':
        weights = np.log(np.array([len(x) for x in arrays], dtype=float))
    else:
        raise ValueError(mode)
    weights /= weights.sum()
    assert np.isclose(weights.sum(), 1.0) and (weights > 0).all()
    return weights

def initial_support(arrays, M_atoms):
    pool = np.vstack(arrays)
    sample_weight = np.concatenate([np.full(len(x), 1.0 / (3.0 * len(x))) for x in arrays])
    model = KMeans(n_clusters=M_atoms, init='k-means++', n_init=1, random_state=0)
    model.fit(pool, sample_weight=sample_weight)
    return model.cluster_centers_

def free_support_barycenter(arrays, M_atoms):
    measures_weights = [np.full(len(x), 1.0 / len(x)) for x in arrays]
    return ot.lp.free_support_barycenter(
        arrays, measures_weights, X_init=initial_support(arrays, M_atoms),
        b=np.full(M_atoms, 1.0 / M_atoms), weights=np.ones(3) / 3.0,
        numItermax=BARYCENTER_MAX_ITER, stopThr=BARYCENTER_TOL,
    )

def sliced_dispersion(arrays, support, lam, seed):
    rng = np.random.default_rng(seed)
    q = np.linspace(0.0, 1.0, N_QUANTILES)
    dimension = arrays[0].shape[1]
    total = 0.0
    for _ in range(N_PROJECTIONS):
        direction = rng.standard_normal(dimension)
        direction /= np.linalg.norm(direction)
        bary_q = np.quantile(support @ direction, q)
        total += sum(
            lam[k] * np.mean((np.quantile(x @ direction, q) - bary_q) ** 2)
            for k, x in enumerate(arrays)
        )
    return float(total / N_PROJECTIONS)

def quantile_1d_dispersion(arrays, lam, seed):
    rng = np.random.default_rng(seed)
    q = np.linspace(0.0, 1.0, N_QUANTILES)
    dimension = arrays[0].shape[1]
    total = 0.0
    for _ in range(N_PROJECTIONS):
        direction = rng.standard_normal(dimension)
        direction /= np.linalg.norm(direction)
        quantiles = [np.quantile(x @ direction, q) for x in arrays]
        bary_q = sum(lam[k] * quantiles[k] for k in range(3))
        total += sum(lam[k] * np.mean((quantiles[k] - bary_q) ** 2) for k in range(3))
    return float(total / N_PROJECTIONS)

def exact_dispersion(arrays, support, lam):
    b = np.full(len(support), 1.0 / len(support))
    total = 0.0
    for k, x in enumerate(arrays):
        a = np.full(len(x), 1.0 / len(x))
        cost = ot.dist(support, x, metric='sqeuclidean')
        total += lam[k] * float(ot.emd2(b, a, cost))
    return float(total)

def estimate_rho(arrays, config, seed):
    lam = frequency_weights(arrays, config['frequency_weights'])
    if config['barycenter'] == 'quantile_1d':
        rho = quantile_1d_dispersion(arrays, lam, seed)
    else:
        support = free_support_barycenter(arrays, int(config['M_atoms']))
        if config['distance'] == 'sliced':
            rho = sliced_dispersion(arrays, support, lam, seed)
        elif config['distance'] == 'exact':
            rho = exact_dispersion(arrays, support, lam)
        else:
            raise ValueError(config['distance'])
    assert np.isfinite(rho) and rho >= 0.0
    return rho, lam

## 5. Resumable rolling computation

Each universe–configuration pair has its own Parquet checkpoint. Compatible complete rows are reloaded; missing dates are computed and appended. Every signal row is accompanied by a matrix-audit row containing the exact asset hash, dimensions, date bounds, and null counts.

In [ ]:
SIGNAL_COLUMNS = [
    'date', 'formation_month', 'universe', 'config_id', 'lookback_months', 'n_assets',
    'M_atoms', 'n_projections', 'n_quantiles', 'frequency_weights', 'scaling',
    'exponent', 'distance', 'barycenter', 'seed', 'lambda_daily', 'lambda_weekly',
    'lambda_monthly', 'rho', 'sqrt_rho', 'engine_version', 'source_signature', 'config_digest'
]

def cache_paths(universe, config_id):
    stem = f'{universe}__{config_id}'
    return CACHE_DIR / f'{stem}.parquet', CACHE_DIR / f'{stem}__audit.parquet'

def config_identity(universe, config):
    payload = dict(config)
    payload.update(
        universe=universe, engine_version=ENGINE_VERSION, base_seed=BASE_SEED,
        n_projections=N_PROJECTIONS, n_quantiles=N_QUANTILES,
        barycenter_max_iter=BARYCENTER_MAX_ITER, barycenter_tol=BARYCENTER_TOL,
        source_signature=SOURCE_SIGNATURES[universe], aggregation='compounded',
    )
    return stable_digest(payload)

def compatible_cache(path, digest):
    if FORCE_RECOMPUTE or not path.exists():
        return pd.DataFrame()
    cached = pd.read_parquet(path)
    required = {'formation_month', 'config_digest'}
    if not required.issubset(cached.columns):
        return pd.DataFrame()
    if cached.empty or not cached['config_digest'].eq(digest).all():
        return pd.DataFrame()
    cached['formation_month'] = pd.to_datetime(cached['formation_month'])
    if 'date' in cached:
        cached['date'] = pd.to_datetime(cached['date'])
    return cached

def asset_hash(assets):
    return hashlib.sha256(','.join(map(str, assets)).encode()).hexdigest()

def compute_one(universe, config, pivot, memberships, month_end):
    config_id = config['config_id']
    digest = config_identity(universe, config)
    signal_path, audit_path = cache_paths(universe, config_id)
    signal_cache = compatible_cache(signal_path, digest)
    audit_cache = compatible_cache(audit_path, digest)
    if signal_cache.empty != audit_cache.empty:
        signal_cache, audit_cache = pd.DataFrame(), pd.DataFrame()

    completed = set(pd.to_datetime(signal_cache['formation_month'])) if not signal_cache.empty else set()
    target_months = sorted(pd.Timestamp(x) for x in memberships)
    stale = completed - set(target_months)
    if stale:
        signal_cache, audit_cache, completed = pd.DataFrame(), pd.DataFrame(), set()

    signal_rows, audit_rows = [], []
    started = time.time()
    print(f'[{universe} | {config_id}] reload={len(completed)} | remaining={len(target_months) - len(completed)}')

    for ordinal, formation_month in enumerate(target_months, start=1):
        if formation_month in completed:
            continue
        assets = memberships[formation_month]
        daily, weekly, monthly, formation_date, start_date = raw_aligned_matrices(
            pivot, assets, formation_month, month_end, int(config['lookback_months'])
        )
        arrays = scale_matrices(daily, weekly, monthly, config)
        seed = seed_for(universe, formation_month)
        rho, lam = estimate_rho(arrays, config, seed)
        common = dict(
            date=formation_date, formation_month=formation_month, universe=universe,
            config_id=config_id, lookback_months=int(config['lookback_months']),
            n_assets=len(assets), M_atoms=int(config['M_atoms']),
            n_projections=N_PROJECTIONS, n_quantiles=N_QUANTILES,
            frequency_weights=config['frequency_weights'], scaling=config['scaling'],
            exponent=config['exponent'], distance=config['distance'], barycenter=config['barycenter'],
            seed=seed, engine_version=ENGINE_VERSION, source_signature=SOURCE_SIGNATURES[universe],
            config_digest=digest,
        )
        signal_rows.append(dict(
            **common, lambda_daily=lam[0], lambda_weekly=lam[1], lambda_monthly=lam[2],
            rho=rho, sqrt_rho=np.sqrt(rho),
        ))
        audit_rows.append(dict(
            **common, start_date=start_date, window_end=formation_date,
            n_daily=len(daily), n_weekly=len(weekly), n_monthly=len(monthly),
            daily_nulls=int(daily.isna().sum().sum()), weekly_nulls=int(weekly.isna().sum().sum()),
            monthly_nulls=int(monthly.isna().sum().sum()), same_columns=True,
            no_future_observations=bool(daily.index.max() <= formation_date),
            asset_hash=asset_hash(assets), first_permno=min(assets), last_permno=max(assets),
        ))

        if WRITE_CACHES and (len(signal_rows) % CHECKPOINT_EVERY == 0):
            current_signal = pd.concat([signal_cache, pd.DataFrame(signal_rows)], ignore_index=True)
            current_audit = pd.concat([audit_cache, pd.DataFrame(audit_rows)], ignore_index=True)
            atomic_parquet(current_signal.sort_values('formation_month'), signal_path)
            atomic_parquet(current_audit.sort_values('formation_month'), audit_path)
            signal_cache, audit_cache = current_signal, current_audit
            signal_rows, audit_rows = [], []
            print(f'  checkpoint {ordinal}/{len(target_months)} | {time.time() - started:.0f}s')
        gc.collect()

    signal = pd.concat([signal_cache, pd.DataFrame(signal_rows)], ignore_index=True)
    audit = pd.concat([audit_cache, pd.DataFrame(audit_rows)], ignore_index=True)
    signal = signal.drop_duplicates(['universe', 'config_id', 'formation_month'], keep='last').sort_values('formation_month')
    audit = audit.drop_duplicates(['universe', 'config_id', 'formation_month'], keep='last').sort_values('formation_month')
    assert len(signal) == len(target_months) == len(audit)
    if WRITE_CACHES:
        atomic_parquet(signal, signal_path)
        atomic_parquet(audit, audit_path)
    print(f'  complete: {len(signal)} months | {time.time() - started:.0f}s')
    return signal, audit

### 5.1. Transaction recovery

Signal and audit checkpoints are separate atomic files. If a kernel stopped between their two writes, only their common completed dates are retained before resumption. This makes recovery deterministic and prevents a partially committed date from being mistaken for a completed calculation.

In [ ]:
for signal_path in sorted(CACHE_DIR.glob('*.parquet')):
    if signal_path.stem.endswith('__audit'):
        continue
    audit_path = signal_path.with_name(signal_path.stem + '__audit.parquet')
    if not audit_path.exists():
        continue
    recovered_signal = pd.read_parquet(signal_path)
    recovered_audit = pd.read_parquet(audit_path)
    if not {'formation_month', 'config_digest'}.issubset(recovered_signal.columns):
        continue
    if not {'formation_month', 'config_digest'}.issubset(recovered_audit.columns):
        continue
    common_dates = set(pd.to_datetime(recovered_signal['formation_month'])) & set(pd.to_datetime(recovered_audit['formation_month']))
    if len(common_dates) != len(recovered_signal) or len(common_dates) != len(recovered_audit):
        recovered_signal = recovered_signal[pd.to_datetime(recovered_signal['formation_month']).isin(common_dates)]
        recovered_audit = recovered_audit[pd.to_datetime(recovered_audit['formation_month']).isin(common_dates)]
        if WRITE_CACHES:
            atomic_parquet(recovered_signal, signal_path)
            atomic_parquet(recovered_audit, audit_path)
        print(f'Recovered checkpoint pair: {signal_path.stem} -> {len(common_dates)} common dates')

## 6. Compute or reload both universes

The loop is intentionally explicit. A completed configuration is loaded immediately; an interrupted configuration resumes from its last atomic checkpoint. Set `FORCE_RECOMPUTE=True` only when a deliberate full rebuild is required.

In [ ]:
all_signals = []
all_audits = []

for universe, source_path in SOURCES.items():
    print(f'\nLoading {universe}: {source_path.name}')
    pivot, memberships, month_end = load_universe(source_path)
    counts = pd.Series({month: len(assets) for month, assets in memberships.items()})
    assert counts.eq(100).all()
    print(f'{len(memberships)} formation months | assets: min={counts.min()}, median={counts.median():.0f}, max={counts.max()}')

    for config in CONFIGS:
        signal_part, audit_part = compute_one(universe, config, pivot, memberships, month_end)
        all_signals.append(signal_part)
        all_audits.append(audit_part)

    del pivot, memberships, month_end
    gc.collect()

## 7. Assemble the candidate and audit artifacts

These files remain under `cache/signal/`: they are calculation products, not trusted article inputs. The validation notebook independently checks their schemas, PIT timing, aligned matrices, deterministic 1-D recomputation, configuration coverage, and numerical invariants before publishing any file under `data/signals/`.

In [ ]:
candidate = pd.concat(all_signals, ignore_index=True).sort_values(['universe', 'config_id', 'date'])
audit = pd.concat(all_audits, ignore_index=True).sort_values(['universe', 'config_id', 'date'])

assert not candidate.duplicated(['universe', 'config_id', 'date']).any()
assert not audit.duplicated(['universe', 'config_id', 'date']).any()
assert np.isfinite(candidate['rho']).all() and candidate['rho'].ge(0).all()
assert np.allclose(candidate['sqrt_rho'] ** 2, candidate['rho'], rtol=1e-12, atol=1e-15)
assert audit[['daily_nulls', 'weekly_nulls', 'monthly_nulls']].eq(0).all().all()
assert audit['same_columns'].all() and audit['no_future_observations'].all()

if WRITE_CACHES:
    atomic_parquet(candidate, CANDIDATE_PATH)
    atomic_parquet(audit, AUDIT_PATH)
    print('Candidate:', CANDIDATE_PATH)
    print('Audit:    ', AUDIT_PATH)
else:
    print('WRITE_CACHES=False — assembled in memory only.')

display(
    candidate.groupby(['universe', 'config_id'])
             .agg(n=('rho', 'size'), date_min=('date', 'min'), date_max=('date', 'max'),
                  n_assets_min=('n_assets', 'min'), n_assets_max=('n_assets', 'max'))
             .reset_index()
)

## 8. Required next step

Run `tests/02_validate_and_publish_pit_signals.ipynb`. The article notebook must consume only the validated Parquet files written by that notebook. A candidate cache must never be substituted directly for a published signal.